# LIMS API Examples

This notebook demonstrates how to use the LIMS API to query data from the Google Sheets-synchronized SQLite database.

## Overview

The LIMS API provides a read-only interface to query experimental data that is automatically synchronized from Google Sheets. The API includes functions for:

- Listing available tables
- Querying tables with filters
- Searching for specific records
- Getting table schemas
- Converting results to pandas DataFrames

All queries automatically exclude deleted rows unless explicitly requested.

In [ ]:
# Import utilities (use util_simple.py for standalone LIMS API usage)
%run util_simple.py

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("\n✓ Notebook setup complete")

## 1. List Available Tables

First, let's see what tables are available in the LIMS database.

In [ ]:
# Get list of all tables
tables = get_lims_tables()

print(f"Found {len(tables)} tables:\n")
for table in tables:
    count = get_table_count(table)
    print(f"  {table}: {count} rows")

## 2. Explore Table Schemas

Let's examine the structure of a table to see what columns are available.

In [ ]:
# Get schema for the Experiments table
schema = get_table_schema('Experiments')

print("Experiments table schema:\n")
for column, sql_type in schema.items():
    print(f"  {column}: {sql_type}")

### Tip: Always Check Column Names

When working with a new table, always check the actual column names first to avoid KeyError exceptions. Column names may differ from what you expect due to:
- Spaces being converted to underscores
- Different naming conventions in the Google Sheet
- Additional suffixes (e.g., `_ID`, `_name`)

Use `get_lims_schema(table_name)` or examine `df.columns` to see available columns.

In [ ]:
# Example: Safe way to work with unknown schemas
# The get_safe_columns() function is now available from util_simple.py

# Get a sample
sample_df = query_lims('Samples', limit=1)

# Safely select columns
safe_cols = get_safe_columns(sample_df, ['Name', 'Strain', 'Strain_name', 'Type', 'Condition'])
print(f"Available columns from preferred list: {safe_cols}")
print(f"\nAll columns in Samples table:")
for col in sample_df.columns:
    if not col.startswith('deleted') and not col.startswith('last_synced') and not col.startswith('row_hash'):
        print(f"  - {col}")

## 3. Query All Records from a Table

Use the helper function `query_lims()` to get all records as a pandas DataFrame.

In [ ]:
# Get all experiments as a DataFrame
experiments_df = query_lims('Experiments')

# Display the DataFrame
print(f"Retrieved {len(experiments_df)} experiments\n")
experiments_df

## 4. Query with Filters

Filter records by specific column values.

In [ ]:
# Get only robotic ALE experiments
robotic_experiments = query_lims(
    'Experiments',
    filters={'Type': 'robotic ALE'}
)

print(f"Found {len(robotic_experiments)} robotic ALE experiments\n")
robotic_experiments

## 5. Select Specific Columns

Query only the columns you need to reduce memory usage.

In [ ]:
# Get only specific columns from Experiments
exp_summary = query_lims(
    'Experiments',
    columns=['Name', 'Type', 'Start_timestamp', 'Description']
)

exp_summary

## 6. Limit Results

Use `limit` to get only the first N rows.

In [ ]:
# Get first 10 strains
strains_sample = query_lims('Strains', limit=10)

print(f"Showing first {len(strains_sample)} strains out of {get_table_count('Strains')} total\n")
strains_sample

## 7. Search for Records

Use the `search_lims()` helper to find records where a column contains specific text (case-insensitive).

In [ ]:
# Search for strains containing "ADP1" in the Name column
adp1_strains = search_lims('Strains', 'Name', 'ADP1')

print(f"Found {len(adp1_strains)} strains with 'ADP1' in the name\n")
adp1_strains.head(10)

## 8. Working with Samples and Measurements

Let's explore how to query related data across tables. First, let's check what columns are available.

In [ ]:
# First, let's see what columns are in the Samples table
samples_schema = get_lims_schema('Samples')

print("Samples table columns:")
for col, sql_type in samples_schema.items():
    if not col.startswith('deleted') and not col.startswith('last_synced') and not col.startswith('row_hash'):
        print(f"  {col}: {sql_type}")

In [ ]:
# Get all samples from a specific experiment
ale1b_samples = query_lims(
    'Samples',
    filters={'Experiment': 'ALE1b'}
)

print(f"Found {len(ale1b_samples)} samples from experiment ALE1b\n")

# Show relevant columns (adjust based on actual schema)
if len(ale1b_samples) > 0:
    display_cols = [col for col in ['Name', 'Strain_name', 'Condition', 'Type', 'Notes'] 
                    if col in ale1b_samples.columns]
    ale1b_samples[display_cols].head(10)
else:
    print("No samples found for experiment ALE1b")

## 9. Combining Queries with Pandas

Use pandas to merge and analyze data from multiple tables.

In [ ]:
# Get measurements for a specific sample
if len(ale1b_samples) > 0:
    sample_name = ale1b_samples.iloc[0]['Name']
    
    measurements = query_lims(
        'Measurements',
        filters={'Sample_ID': sample_name}
    )
    
    if len(measurements) > 0:
        print(f"Measurements for sample {sample_name}:\n")
        # Show relevant columns
        display_cols = [col for col in ['Name', 'Type', 'Timestamp', 'Data', 'Protocol'] 
                        if col in measurements.columns]
        display(measurements[display_cols].head(10))
    else:
        print(f"No measurements found for sample {sample_name}")
        print("\nTrying to find any measurements...")
        all_measurements = query_lims('Measurements', limit=5)
        if len(all_measurements) > 0:
            print(f"\nFound {get_table_count('Measurements')} total measurements")
            display(all_measurements.head())
else:
    print("No samples found to query measurements")

## 10. Advanced: Direct API Usage

For more control, you can use the raw LIMS API functions directly.

In [ ]:
# Use the raw API for more complex queries
from aisynbiopipeline.limsapi import query_table

# Query with ordering
results = query_table(
    'Genes',
    columns=['Locus_tag', 'Name', 'Function'],
    order_by='Locus_tag',
    order_desc=False,
    limit=20
)

# Convert to DataFrame
genes_df = pd.DataFrame(results)
print(f"First 20 genes ordered by locus tag:\n")
genes_df

## 11. Analyzing Measurement Data

Let's do a more complex analysis combining multiple tables.

In [ ]:
# Get all measurements
all_measurements = query_lims('Measurements')

# Get measurement types if that table exists
try:
    measurement_types = query_lims('Measurement_types')
    has_types_table = True
except:
    has_types_table = False
    print("Measurement_types table not available")

print(f"Total measurements: {len(all_measurements)}")

# Show measurement type distribution
if 'Type' in all_measurements.columns:
    print(f"\nMeasurement types distribution:")
    print(all_measurements['Type'].value_counts())

# Show a sample of the data
print(f"\nSample of measurement data:")
display_cols = [col for col in ['Sample_ID', 'Name', 'Type', 'Data', 'Timestamp'] 
                if col in all_measurements.columns]
all_measurements[display_cols].head(10)

## 12. Summary and Best Practices

### Key Functions

- **`get_lims_tables()`** - List all available tables
- **`query_lims(table, filters, columns, limit)`** - Query a table and return a DataFrame
- **`search_lims(table, column, search_term)`** - Search for records containing text
- **`get_table_schema(table)`** - Get the schema/structure of a table
- **`get_table_count(table)`** - Count rows in a table

### Best Practices

1. **Start with schema exploration**: Use `get_table_schema()` to understand what columns are available
2. **Use filters**: Filter data at the database level rather than in pandas for better performance
3. **Select specific columns**: Only query the columns you need
4. **Use limits for exploration**: When exploring large tables, use `limit` to get a sample first
5. **Leverage pandas**: Use pandas for complex joins, aggregations, and visualizations after querying

### Notes

- All queries automatically exclude deleted rows (soft deletes)
- Column names with spaces or hyphens are converted to underscores in the database
- The database is read-only - no modifications allowed through the API
- Data is automatically synced from Google Sheets every 10 minutes (if daemon is running)